# 6 — Causal Ablation Validation

**Purpose:** Validate that concept-only neurons (A\B set) are causally necessary for
concept prediction by measuring the drop in log P(keyword) when they are ablated.

**Protocol:** For each of 10 target concepts, at each of 28 layers independently:
1. Run clean forward passes on 50 prompts containing the concept
2. Ablate concept-only neurons (zero or mean replacement) and re-run
3. Compare Δ log P against shared-neuron and random-neuron controls

**Design doc:** See `6_ablation_design.md` in this directory.

In [ ]:
# Cell 1 – Dependency setup & configuration
import subprocess, sys, os

pkgs = ["h5py", "transformers", "torch", "numpy", "pandas", "tqdm",
        "pyarrow", "accelerate", "scipy", "matplotlib", "seaborn"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── Constants ──
MODEL = "QW"

LANG = "P"
PREFIX = f"{LANG}_{MODEL}_"
MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}
MODEL_ID = MODEL_CONFIGS[MODEL]["id"]
N_LAYERS = MODEL_CONFIGS[MODEL]["n_layers"]
MLP_DIM = MODEL_CONFIGS[MODEL]["mlp_dim"]          # MLP output dimension (post-down_proj), NOT intermediate 18944
N_PROMPTS = None          # None = use all available prompts per concept
BATCH_SIZE = 8           # prompts per forward pass
SEED = 42
CHECKPOINT_EVERY = 2     # save partial results every N concepts

# ── Target concepts: {csv_object_name: keyword_string} ──
CONCEPTS = {
    # High concept-fraction ASTs
    "ast__Import":      "import",
    "ast__Assert":      "assert",
    "ast__Break":       "break",
    # Mid concept-fraction ASTs
    "ast__For":         "for",
    "ast__While":       "while",
    "ast__Try":         "try",
    # Low concept-fraction ASTs
    "ast__If":          "if",
    "ast__FunctionDef": "def",
    "ast__Lambda":      "lambda",
    # Builtin negative controls
    "builtin__len":     "len",
    "builtin__range":   "range",
}

ABLATION_MODES = ["zero", "mean"]

# ── Paths ──
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/New-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/New-Atlas/src"
SRC_PATH = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Neuron list CSVs
NEURON_CSV_STRICT = f"{PREFIX}4_neuron_list_eps0.5_cons0.8_all_layers_both.csv"
PROMPT_PARQUET = f"{PREFIX}1A_object_prompts.parquet"

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Data dir: {DATA_DIR}")
print(f"Concepts: {len(CONCEPTS)}")
print(f"Ablation modes: {ABLATION_MODES}")

In [ ]:
# Cell 2 – Imports & model loading
import ast as ast_mod
import numpy as np
import pandas as pd
import torch
import logging
import re
import json
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("notebook6")

from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
).eval()

print(f"Model loaded | Params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {DEVICE} | dtype: float16")

In [ ]:
# Cell 3 – Tokenizer verification: map each concept keyword to a single token ID
target_token_ids = {}

for concept, keyword in CONCEPTS.items():
    ids = tokenizer.encode(keyword, add_special_tokens=False)
    target_token_ids[concept] = ids[0]
    decoded = tokenizer.decode([ids[0]])
    is_single = len(ids) == 1
    status = "OK" if is_single else f"MULTI-TOKEN ({len(ids)} tokens, using first)"
    print(f"  {concept:25s} keyword='{keyword:8s}' -> id={ids[0]:6d} decoded='{decoded}' {status}")

print(f"\nAll {len(target_token_ids)} keywords mapped.")

In [ ]:
# Cell 4 – Load neuron lists from CSV
csv_path = os.path.join(DATA_DIR, NEURON_CSV_STRICT)
df_neurons = pd.read_csv(csv_path)
print(f"Loaded neuron list: {len(df_neurons)} rows from {NEURON_CSV_STRICT}")

# Parse stringified lists and build lookup dict
# Key: (concept, layer) -> {"concept_only": [...], "both": [...], "token_only": [...]}
neuron_data = {}

for concept in CONCEPTS:
    sub = df_neurons[df_neurons["object"] == concept]
    if len(sub) == 0:
        print(f"  WARNING: {concept} not found in CSV")
        continue
    for _, row in sub.iterrows():
        lid = int(row["layer"])
        neuron_data[(concept, lid)] = {
            "concept_only": ast_mod.literal_eval(row["concept_only"]),
            "both":         ast_mod.literal_eval(row["both"]),
            "token_only":   ast_mod.literal_eval(row["token_only"]),
        }

# Summary
for concept in CONCEPTS:
    layers_present = sum(1 for l in range(N_LAYERS) if (concept, l) in neuron_data)
    avg_co = np.mean([len(neuron_data[(concept, l)]["concept_only"])
                      for l in range(N_LAYERS) if (concept, l) in neuron_data])
    print(f"  {concept:25s} layers={layers_present:2d}  mean concept_only={avg_co:6.1f}")

In [ ]:
# Cell 5 – Load & sample prompts
rng = np.random.default_rng(SEED)

parquet_path = os.path.join(DATA_DIR, PROMPT_PARQUET)
df_prompts = pd.read_parquet(parquet_path)
print(f"Loaded prompts: {len(df_prompts)} rows")

concept_prompts = {}  # concept -> list of prompt strings

for concept, keyword in CONCEPTS.items():
    prefix, name = concept.split("__", 1)
    if prefix == "ast":
        pool = df_prompts[df_prompts["ast_node"] == name]
    else:
        pool = df_prompts[df_prompts["builtin_obj"] == name]

    if N_PROMPTS is not None:
        sampled = pool.sample(n=min(N_PROMPTS, len(pool)), random_state=SEED)
    else:
        sampled = pool.sample(frac=1, random_state=SEED)  # shuffle all
    concept_prompts[concept] = sampled["prompt_text"].tolist()
    print(f"  {concept:25s} pool={len(pool):6d}  using={len(sampled)}")

# Free the large dataframe
del df_prompts
print("Prompt sampling complete.")

In [ ]:
# Cell 6 – AblationHook class

class AblationHook:
    """
    Forward hook that ablates specific neuron indices in the MLP output
    at a single layer. Supports zero-ablation and mean-ablation.

    The hook modifies the MLP module's output tensor (shape [batch, seq, 3584]),
    which is the post-down_proj contribution to the residual stream.
    """

    def __init__(self, model, layer_id, neuron_indices, mode="zero",
                 mean_activations=None):
        """
        Args:
            model: the HF model
            layer_id: which layer (0-27)
            neuron_indices: list of int indices to ablate (dim 2 of MLP output)
            mode: "zero" or "mean"
            mean_activations: tensor of shape [3584] with per-dim means (for mode="mean")
        """
        self.layer_id = layer_id
        self.neuron_indices = neuron_indices
        self.mode = mode
        self.mean_activations = mean_activations
        self.handle = None

        # Locate the MLP module (same pattern as ActivationExtractor)
        module_dict = dict(model.named_modules())
        mlp_name = f"model.layers.{layer_id}.mlp"
        if mlp_name not in module_dict:
            raise ValueError(f"MLP module not found: {mlp_name}")
        self.module = module_dict[mlp_name]

    def _hook_fn(self, module, input, output):
        if isinstance(output, tuple):
            out = output[0]
        else:
            out = output
        # out shape: [batch, seq_len, mlp_dim]
        modified = out.clone()
        if self.mode == "zero":
            modified[:, :, self.neuron_indices] = 0.0
        elif self.mode == "mean" and self.mean_activations is not None:
            modified[:, :, self.neuron_indices] = self.mean_activations[self.neuron_indices]
        else:
            modified[:, :, self.neuron_indices] = 0.0  # fallback to zero
        if isinstance(output, tuple):
            return (modified,) + output[1:]
        return modified

    def register(self):
        self.handle = self.module.register_forward_hook(self._hook_fn)
        return self

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None


# Sanity check: register and remove a hook on layer 0
test_hook = AblationHook(model, 0, [0, 1, 2], mode="zero")
test_hook.register()
test_hook.remove()
print("AblationHook OK")

In [ ]:
# Cell 7 – Log-probability extraction

def find_target_positions(input_ids, target_token_id):
    """
    For each sequence in a batch, find the first occurrence of target_token_id.
    Returns list of positions (or None if not found).
    The prediction position is target_pos - 1 (autoregressive).
    """
    batch_size = input_ids.shape[0]
    positions = []
    for b in range(batch_size):
        ids = input_ids[b].tolist()
        found = None
        for i, tid in enumerate(ids):
            if tid == target_token_id and i > 0:
                found = i
                break
        positions.append(found)
    return positions


def get_logprobs_batch(model, tokenizer, prompts, target_token_id, device):
    """
    Run a batched forward pass and extract log P(target_token) for each prompt.

    Returns:
        list of (log_prob, target_pos) tuples. log_prob is None if keyword not found.
    """
    encoded = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, add_special_tokens=False,
    ).to(device)

    input_ids = encoded["input_ids"]
    positions = find_target_positions(input_ids, target_token_id)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=encoded["attention_mask"])
        logits = outputs.logits  # [batch, seq_len, vocab_size]

    results = []
    for b, pos in enumerate(positions):
        if pos is None:
            results.append((None, None))
        else:
            # Position pos-1 predicts token at pos
            log_probs = torch.log_softmax(logits[b, pos - 1, :], dim=-1)
            lp = log_probs[target_token_id].item()
            results.append((lp, pos))

    return results


# Quick sanity check
test_prompts = concept_prompts["ast__Assert"][:2]
test_results = get_logprobs_batch(model, tokenizer, test_prompts,
                                   target_token_ids["ast__Assert"], DEVICE)
for i, (lp, pos) in enumerate(test_results):
    if lp is not None:
        print(f"  Prompt {i}: log P('assert') = {lp:.4f} at position {pos}")
    else:
        print(f"  Prompt {i}: keyword not found in token sequence (skipped)")
print("Log-probability extraction OK")

In [ ]:
# Cell 8 – Control condition sampling

def sample_shared_neurons(both_indices, n_needed, rng):
    """Sample n_needed neurons from the shared (A∩B) set."""
    if len(both_indices) <= n_needed:
        return list(both_indices)
    return rng.choice(both_indices, size=n_needed, replace=False).tolist()


def sample_random_neurons(concept_only, both, token_only, n_needed, mlp_dim, rng):
    """Sample n_needed neurons from outside all three sets."""
    excluded = set(concept_only) | set(both) | set(token_only)
    available = [i for i in range(mlp_dim) if i not in excluded]
    if len(available) <= n_needed:
        return available
    return rng.choice(available, size=n_needed, replace=False).tolist()


print("Control sampling functions defined.")

In [ ]:
# Cell 9 – Compute mean activations for mean-ablation mode
from module2.extraction import ActivationExtractor

# Detect MLP hook pattern
mlp_pattern = None
for name, _ in model.named_modules():
    m = re.match(r'^(.*?)(\d+)(\.mlp)$', name)
    if m:
        if mlp_pattern is None:
            mlp_pattern = m.group(1) + "{layer_id}" + m.group(3)
            break
print(f"MLP pattern: {mlp_pattern}")

# Collect mean MLP outputs across ~200 random prompts
all_prompts_flat = [p for ps in concept_prompts.values() for p in ps]
mean_sample = rng.choice(all_prompts_flat, size=min(200, len(all_prompts_flat)), replace=False).tolist()

extractor = ActivationExtractor(model=model, tokenizer=tokenizer, device=DEVICE, n_layers=N_LAYERS)
extractor.set_hook_pattern(mlp_pattern)
extractor.register_hooks()

act_sums = {}  # layer_id -> running sum tensor
n_collected = 0

for prompt in tqdm(mean_sample, desc="Computing mean activations"):
    acts = extractor.extract(prompt, token_pos=-1)
    for lid, vec in acts.items():
        if lid not in act_sums:
            act_sums[lid] = torch.zeros_like(vec)
        act_sums[lid] += vec
    n_collected += 1

extractor.remove_hooks()

mean_activations = {}  # layer_id -> tensor on device
for lid, s in act_sums.items():
    mean_activations[lid] = (s / n_collected).to(dtype=torch.float16).to(DEVICE)

del act_sums
print(f"Mean activations computed from {n_collected} prompts across {len(mean_activations)} layers.")
# Show stats for layer 14 as sanity check
m14 = mean_activations.get(14)
if m14 is not None:
    print(f"  Layer 14: shape={tuple(m14.shape)}, mean={m14.mean().item():.6f}, std={m14.std().item():.6f}")

In [ ]:
# Cell 10 – Main ablation loop
import time

results = []
ablation_rng = np.random.default_rng(SEED)
start_time = time.time()

concept_list = list(CONCEPTS.items())

for concept_idx, (concept, keyword) in enumerate(concept_list):
    prompts = concept_prompts[concept]
    target_tid = target_token_ids[concept]
    print(f"\n[{concept_idx+1}/{len(concept_list)}] {concept} (keyword='{keyword}', {len(prompts)} prompts)")

    for layer_id in tqdm(range(N_LAYERS), desc=f"  Layers", leave=False):
        key = (concept, layer_id)
        if key not in neuron_data:
            continue

        nd = neuron_data[key]
        concept_only = nd["concept_only"]
        both = nd["both"]
        token_only = nd["token_only"]
        n_ablate = len(concept_only)

        if n_ablate == 0:
            continue

        # Sample control conditions (matched neuron count)
        shared_sample = sample_shared_neurons(both, n_ablate, ablation_rng)
        random_sample = sample_random_neurons(concept_only, both, token_only,
                                               n_ablate, MLP_DIM, ablation_rng)

        conditions = {
            "concept_only": concept_only,
            "shared":       shared_sample,
            "random":       random_sample,
        }

        # ── Clean forward pass (shared across all conditions) ──
        clean_data = []  # list of (prompt_idx, log_prob, target_pos)
        for batch_start in range(0, len(prompts), BATCH_SIZE):
            batch = prompts[batch_start:batch_start + BATCH_SIZE]
            batch_results = get_logprobs_batch(model, tokenizer, batch, target_tid, DEVICE)
            for i, (lp, pos) in enumerate(batch_results):
                if lp is not None:
                    clean_data.append((batch_start + i, lp, pos))

        if len(clean_data) == 0:
            continue

        # ── Ablated forward passes ──
        for ablation_mode in ABLATION_MODES:
            mean_act = mean_activations.get(layer_id) if ablation_mode == "mean" else None

            for cond_name, cond_indices in conditions.items():
                if len(cond_indices) == 0:
                    continue

                hook = AblationHook(model, layer_id, cond_indices,
                                    mode=ablation_mode,
                                    mean_activations=mean_act)
                hook.register()

                ablated_logps = {}  # prompt_idx -> log_prob
                for batch_start in range(0, len(prompts), BATCH_SIZE):
                    batch = prompts[batch_start:batch_start + BATCH_SIZE]
                    batch_results = get_logprobs_batch(model, tokenizer, batch,
                                                       target_tid, DEVICE)
                    for i, (lp, pos) in enumerate(batch_results):
                        if lp is not None:
                            ablated_logps[batch_start + i] = lp

                hook.remove()

                # Compute deltas (only for prompts present in both clean and ablated)
                deltas = []
                for pidx, clean_lp, _ in clean_data:
                    if pidx in ablated_logps:
                        deltas.append(clean_lp - ablated_logps[pidx])

                if len(deltas) == 0:
                    continue

                results.append({
                    "concept":        concept,
                    "keyword":        keyword,
                    "layer":          layer_id,
                    "condition":      cond_name,
                    "ablation_mode":  ablation_mode,
                    "n_ablated":      len(cond_indices),
                    "n_prompts":      len(deltas),
                    "mean_delta":     float(np.mean(deltas)),
                    "std_delta":      float(np.std(deltas)),
                    "median_delta":   float(np.median(deltas)),
                    "deltas_json":    json.dumps(deltas),
                })

    # Checkpoint
    if (concept_idx + 1) % CHECKPOINT_EVERY == 0:
        ckpt_path = os.path.join(DATA_DIR, f"{PREFIX}6_ablation_ckpt_{concept_idx+1}.csv")
        pd.DataFrame([{k: v for k, v in r.items() if k != "deltas_json"}
                       for r in results]).to_csv(ckpt_path, index=False)
        elapsed = time.time() - start_time
        print(f"  Checkpoint saved ({concept_idx+1}/{len(concept_list)}, {elapsed/60:.1f} min)")

elapsed = time.time() - start_time
print(f"\nAblation complete: {len(results)} result rows in {elapsed/60:.1f} min")

In [ ]:
# Cell 11 – Statistical tests
from scipy import stats

# Build a lookup for raw deltas
delta_lookup = {}  # (concept, layer, condition, mode) -> list of deltas
for r in results:
    key = (r["concept"], r["layer"], r["condition"], r["ablation_mode"])
    delta_lookup[key] = json.loads(r["deltas_json"])

stat_results = []

for ablation_mode in ABLATION_MODES:
    for concept in CONCEPTS:
        for layer_id in range(N_LAYERS):
            d_concept = delta_lookup.get((concept, layer_id, "concept_only", ablation_mode))
            d_shared  = delta_lookup.get((concept, layer_id, "shared", ablation_mode))
            d_random  = delta_lookup.get((concept, layer_id, "random", ablation_mode))

            if d_concept is None or d_random is None:
                continue

            # Paired t-test: concept_only > random (one-sided)
            if len(d_concept) >= 2 and len(d_random) >= 2:
                min_len = min(len(d_concept), len(d_random))
                t_stat, t_pval = stats.ttest_rel(
                    d_concept[:min_len], d_random[:min_len], alternative="greater"
                )
            else:
                t_stat, t_pval = np.nan, np.nan

            # Wilcoxon signed-rank (one-sided)
            try:
                min_len = min(len(d_concept), len(d_random))
                diffs = [c - r for c, r in zip(d_concept[:min_len], d_random[:min_len])]
                w_stat, w_pval = stats.wilcoxon(diffs, alternative="greater")
            except (ValueError, ZeroDivisionError):
                w_stat, w_pval = np.nan, np.nan

            stat_results.append({
                "ablation_mode":    ablation_mode,
                "concept":          concept,
                "layer":            layer_id,
                "mean_delta_concept": float(np.mean(d_concept)),
                "mean_delta_shared":  float(np.mean(d_shared)) if d_shared else np.nan,
                "mean_delta_random":  float(np.mean(d_random)),
                "effect_size":      float(np.mean(d_concept) - np.mean(d_random)),
                "t_stat":           t_stat,
                "t_pval":           t_pval,
                "w_stat":           w_stat,
                "w_pval":           w_pval,
                # Bonferroni correction for 28 layers
                "t_pval_bonf":      min(t_pval * N_LAYERS, 1.0) if not np.isnan(t_pval) else np.nan,
                "w_pval_bonf":      min(w_pval * N_LAYERS, 1.0) if not np.isnan(w_pval) else np.nan,
            })

df_stats = pd.DataFrame(stat_results)
print(f"Statistical tests: {len(df_stats)} rows")

# Summary: how many concept/layer combos are significant (Bonferroni-corrected Wilcoxon)?
for mode in ABLATION_MODES:
    sub = df_stats[df_stats["ablation_mode"] == mode]
    n_sig = (sub["w_pval_bonf"] < 0.05).sum()
    n_total = len(sub)
    print(f"  {mode}: {n_sig}/{n_total} concept/layer combos significant (p<0.05 Bonferroni)")

In [ ]:
# Cell 12 – Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

concept_order = list(CONCEPTS.keys())

# ── Figure 1: Heatmap (zero-ablation effect size) ──
fig, ax = plt.subplots(figsize=(14, 5))
heatmap_data = np.full((len(concept_order), N_LAYERS), np.nan)

for i, concept in enumerate(concept_order):
    for layer in range(N_LAYERS):
        row = df_stats[(df_stats["concept"] == concept) &
                       (df_stats["layer"] == layer) &
                       (df_stats["ablation_mode"] == "zero")]
        if len(row) > 0:
            heatmap_data[i, layer] = row.iloc[0]["effect_size"]

sns.heatmap(heatmap_data, ax=ax, cmap="YlOrRd", center=0,
            xticklabels=range(N_LAYERS),
            yticklabels=[c.replace("ast__", "").replace("builtin__", "") for c in concept_order])
ax.set_xlabel("Layer")
ax.set_ylabel("Concept")
ax.set_title("Effect size (Δ_concept − Δ_random) — zero-ablation")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, f"{PREFIX}6_fig1_heatmap.png"), dpi=150)
plt.show()


# ── Figure 2: Layer profile for top-3 ASTs ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
top3 = ["ast__Import", "ast__Assert", "ast__Break"]

for ax, concept in zip(axes, top3):
    for cond, color, ls in [("concept_only", "#d62728", "-"),
                             ("shared", "#1f77b4", "--"),
                             ("random", "#7f7f7f", ":")]:
        layers_x, means_y = [], []
        for layer in range(N_LAYERS):
            key = (concept, layer, cond, "zero")
            if key in delta_lookup:
                layers_x.append(layer)
                means_y.append(np.mean(delta_lookup[key]))
        ax.plot(layers_x, means_y, color=color, linestyle=ls, label=cond, linewidth=1.5)

    ax.set_title(concept.replace("ast__", ""))
    ax.set_xlabel("Layer")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Mean Δ log P")
fig.suptitle("Layer profile — zero-ablation", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, f"{PREFIX}6_fig2_layer_profile.png"), dpi=150)
plt.show()


# ── Figure 3: Bar chart at layer 14 ──
fig, ax = plt.subplots(figsize=(12, 5))
bar_data = []

for concept in concept_order:
    for cond in ["concept_only", "shared", "random"]:
        key = (concept, 14, cond, "zero")
        if key in delta_lookup:
            sig_row = df_stats[(df_stats["concept"] == concept) &
                               (df_stats["layer"] == 14) &
                               (df_stats["ablation_mode"] == "zero")]
            sig = ""
            if cond == "concept_only" and len(sig_row) > 0:
                p = sig_row.iloc[0]["w_pval_bonf"]
                if p < 0.001: sig = "***"
                elif p < 0.01: sig = "**"
                elif p < 0.05: sig = "*"
            bar_data.append({
                "concept": concept.replace("ast__", "").replace("builtin__", ""),
                "condition": cond,
                "mean_delta": np.mean(delta_lookup[key]),
                "sig": sig,
            })

df_bar = pd.DataFrame(bar_data)
if len(df_bar) > 0:
    pivot = df_bar.pivot(index="concept", columns="condition", values="mean_delta")
    pivot = pivot.reindex([c.replace("ast__", "").replace("builtin__", "") for c in concept_order])
    pivot[["concept_only", "shared", "random"]].plot(kind="bar", ax=ax,
        color=["#d62728", "#1f77b4", "#7f7f7f"])
    ax.set_ylabel("Mean Δ log P")
    ax.set_title("Layer 14 — zero-ablation")
    ax.legend(title="Condition")
    # Add significance stars
    for i, concept_short in enumerate(pivot.index):
        sig_rows = [r for r in bar_data if r["concept"] == concept_short and r["sig"]]
        if sig_rows:
            val = pivot.loc[concept_short, "concept_only"]
            if not np.isnan(val):
                ax.text(i - 0.25, val + 0.01, sig_rows[0]["sig"], ha="center", fontsize=10)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, f"{PREFIX}6_fig3_bar_layer14.png"), dpi=150)
plt.show()


# ── Figure 4: Zero vs Mean ablation comparison ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, concept in zip(axes, top3):
    for mode, color, ls in [("zero", "#d62728", "-"), ("mean", "#ff7f0e", "--")]:
        layers_x, means_y = [], []
        for layer in range(N_LAYERS):
            key = (concept, layer, "concept_only", mode)
            if key in delta_lookup:
                layers_x.append(layer)
                means_y.append(np.mean(delta_lookup[key]))
        ax.plot(layers_x, means_y, color=color, linestyle=ls, label=mode, linewidth=1.5)

    ax.set_title(concept.replace("ast__", ""))
    ax.set_xlabel("Layer")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Mean Δ log P (concept_only)")
fig.suptitle("Zero vs Mean ablation — concept-only neurons", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, f"{PREFIX}6_fig4_zero_vs_mean.png"), dpi=150)
plt.show()

print("All figures saved.")

In [ ]:
# Cell 13 – Save results

# Main results (without raw deltas)
df_results = pd.DataFrame([{k: v for k, v in r.items() if k != "deltas_json"}
                            for r in results])
results_path = os.path.join(DATA_DIR, f"{PREFIX}6_ablation_results.csv")
df_results.to_csv(results_path, index=False)
print(f"Saved: {results_path} ({len(df_results)} rows)")

# Full results with raw deltas (for reproducibility)
df_results_full = pd.DataFrame(results)
full_path = os.path.join(DATA_DIR, f"{PREFIX}6_ablation_results_full.csv")
df_results_full.to_csv(full_path, index=False)
print(f"Saved: {full_path} ({len(df_results_full)} rows, with raw deltas)")

# Statistical tests
stats_path = os.path.join(DATA_DIR, f"{PREFIX}6_ablation_stats.csv")
df_stats.to_csv(stats_path, index=False)
print(f"Saved: {stats_path} ({len(df_stats)} rows)")

# Clean up checkpoint files
import glob
ckpts = glob.glob(os.path.join(DATA_DIR, f"{PREFIX}6_ablation_ckpt_*.csv"))
if ckpts:
    print(f"\nFound {len(ckpts)} checkpoint files (keep for now, delete manually if desired)")

In [ ]:
# Cell 14 – Summary & cleanup

print("=" * 70)
print("ABLATION SUMMARY")
print("=" * 70)

for mode in ABLATION_MODES:
    print(f"\n── {mode.upper()} ABLATION ──")
    sub = df_stats[df_stats["ablation_mode"] == mode].copy()
    if len(sub) == 0:
        print("  No results")
        continue

    # Per-concept summary
    for concept in concept_order:
        csub = sub[sub["concept"] == concept]
        if len(csub) == 0:
            continue
        n_sig = (csub["w_pval_bonf"] < 0.05).sum()
        peak_layer = csub.loc[csub["effect_size"].idxmax(), "layer"] if len(csub) > 0 else "N/A"
        peak_effect = csub["effect_size"].max()
        mean_effect = csub["effect_size"].mean()
        label = concept.replace("ast__", "").replace("builtin__", "")
        print(f"  {label:15s}  sig_layers={n_sig:2d}/28  peak_layer={int(peak_layer):2d}  "
              f"peak_effect={peak_effect:.4f}  mean_effect={mean_effect:.4f}")

print("\n" + "=" * 70)
print(f"Total runtime: {elapsed/60:.1f} minutes")
print(f"Output files: 6_ablation_results.csv, 6_ablation_stats.csv")
print(f"Figures: 6_fig1_heatmap.png, 6_fig2_layer_profile.png, "
      f"6_fig3_bar_layer14.png, 6_fig4_zero_vs_mean.png")
print("\n6 complete.")

In [ ]:
# Cell 15 – Disconnect Colab runtime
try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab — skipping unassign.")
